# 교차 계정: AWS RAM을 통해 Private API 공유

이 실습에서는 자체 관리형 VPC Lattice와 [AWS Resource Access Manager(RAM)](https://docs.aws.amazon.com/ram/latest/userguide/what-is.html)을 사용하여 **계정 A**의 Amazon Bedrock AgentCore Gateway를 **계정 B의 Private API Gateway**에 연결합니다.

이는 교차 계정 연결을 위한 표준 엔터프라이즈 패턴입니다. 리소스 소유자(계정 B)가 VPC Lattice 리소스를 생성하고 RAM을 통해 Gateway 소유자(계정 A)와 공유합니다.

## 아키텍처

![RAM을 통한 교차 계정 대상 구성](./images/ram-target.png)

## 작동 방식

1. **계정 B**(리소스 소유자): Private API Gateway + VPCE + VPC Lattice Resource Gateway + Resource Configuration
2. **계정 B**가 AWS RAM을 통해 Resource Configuration을 **계정 A**와 공유합니다.
3. **계정 A**가 RAM 공유를 수락하고 공유된 Resource Configuration ARN을 확인합니다.
4. **계정 A**가 공유된 Resource Configuration ARN을 가리키는 `selfManagedLatticeResource`로 Gateway 대상을 생성합니다.
5. AgentCore가 Resource Configuration을 서비스 네트워크에 연결하여 교차 계정 연결을 완성합니다.

![RAM 리소스 공유](./images/ram.png)

자체 관리형 Lattice에 대한 배경 정보는 [자체 관리형 Lattice README](./README.md)를 참조하세요.

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(us-west-2 리전의 VPC + AgentCore Gateway 배포)
- **계정 B 구성 완료**: 실습 0에서 계정 B 설정(자격 증명, 부트스트랩, VPC 배포) 셀의 주석을 해제하고 실행

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 계정 A 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES

# 실습 0에서 계정 B 변수 복원
%store -r ACCOUNT_B_ID
%store -r ACCOUNT_B_PROFILE
%store -r VPC_ACCB_USW2_ID
%store -r VPC_ACCB_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID
os.environ["ACCOUNT_B_ID"] = ACCOUNT_B_ID

REGION = "us-west-2"
session_a = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
session_b = boto3.Session(profile_name=ACCOUNT_B_PROFILE, region_name=REGION)

agentcore = session_a.client("bedrock-agentcore-control")
lattice_b = session_b.client("vpc-lattice")
ram_a = session_a.client("ram")
ram_b = session_b.client("ram")

# Cognito 클라이언트 보안 암호 가져오기
cognito = session_a.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account A: {ACCOUNT_A_ID} (gateway owner)")
print(f"Account B: {ACCOUNT_B_ID} (resource owner)")
print(f"Gateway:   {GATEWAY_ID}")
print(f"VPC (B):   {VPC_ACCB_USW2_ID}")

## 2단계: 계정 B에 Private API Gateway 배포

계정 B의 VPC에 모의 통합이 적용된 Private API Gateway를 배포합니다. 다른 실습에서 사용한 것과 동일한 `PrivateApigwStack` 패턴으로, API 키로 보호되는 모의 `/health` 및 `/items` 엔드포인트를 구성합니다.

In [ ]:
!ACCOUNT_B_ID={ACCOUNT_B_ID} cdk deploy CrossAccountApigw-AccountB \
    --profile {ACCOUNT_B_PROFILE} \
    --require-approval never \
    --outputs-file cross-account-outputs.json

In [ ]:
with open("cross-account-outputs.json") as f:
    outputs = json.load(f)

apigw_b = outputs["CrossAccountApigw-AccountB"]
CROSSACCT_API_ID = apigw_b["ApiId"]
CROSSACCT_API_KEY_ID = apigw_b["ApiKeyId"]
CROSSACCT_VPCE_ID = apigw_b["VpceId"]
CROSSACCT_VPCE_SG_ID = apigw_b["VpceSgId"]

# 계정 B의 API-VPCE DNS
API_VPCE_DNS = f"{CROSSACCT_API_ID}-{CROSSACCT_VPCE_ID}.execute-api.{REGION}.amazonaws.com"

# 계정 B에서 API 키 값 가져오기
apigw_client_b = session_b.client("apigateway")
api_key_response = apigw_client_b.get_api_key(apiKey=CROSSACCT_API_KEY_ID, includeValue=True)
API_KEY_VALUE = api_key_response["value"]

print("=== Account B Private API Gateway ===")
print(f"VPC ID:       {VPC_ACCB_USW2_ID}")
print(f"API ID:       {CROSSACCT_API_ID}")
print(f"VPCE ID:      {CROSSACCT_VPCE_ID}")
print(f"API-VPCE DNS: {API_VPCE_DNS}")
print(f"VPCE SG:      {CROSSACCT_VPCE_SG_ID}")

## 3단계: 계정 B에 VPC Lattice 리소스 생성

리소스 소유자인 계정 B에서 다음을 생성합니다.
1. **Resource Gateway**: 계정 B의 VPC에 ENI를 프로비저닝합니다.
2. **Resource Configuration**: AgentCore가 접근할 수 있는 엔드포인트(API-VPCE DNS)를 정의합니다.

![Resource Gateway 구성](./images/resource-gateway.png)

In [ ]:
# 계정 B에 Resource Gateway 생성
rg_response = lattice_b.create_resource_gateway(
    name="cross-account-rg",
    vpcIdentifier=VPC_ACCB_USW2_ID,
    subnetIds=VPC_ACCB_USW2_PRIVATE_SUBNETS,
    securityGroupIds=[CROSSACCT_VPCE_SG_ID],
    ipAddressType="IPV4",
)

RESOURCE_GATEWAY_ID = rg_response["id"]
print(f"Resource Gateway ID:  {RESOURCE_GATEWAY_ID}")
print(f"Status:               {rg_response['status']}")

In [ ]:
# Resource Gateway가 ACTIVE 상태가 될 때까지 대기
while True:
    rg = lattice_b.get_resource_gateway(resourceGatewayIdentifier=RESOURCE_GATEWAY_ID)
    status = rg["status"]
    print(f"Status: {status}")
    if status == "ACTIVE":
        print("\nResource Gateway is active!")
        break
    if status == "CREATE_FAILED":
        print(f"\nFailed: {rg}")
        break
    time.sleep(15)

In [ ]:
# 계정 B에 Resource Configuration 생성
rc_response = lattice_b.create_resource_configuration(
    name="cross-account-rc",
    type="SINGLE",
    resourceGatewayIdentifier=RESOURCE_GATEWAY_ID,
    resourceConfigurationDefinition={
        "dnsResource": {
            "domainName": API_VPCE_DNS,
            "ipAddressType": "IPV4",
        }
    },
    portRanges=["443"],
)

RESOURCE_CONFIG_ARN = rc_response["arn"]
RESOURCE_CONFIG_ID = rc_response["id"]
print(f"Resource Configuration ID:  {RESOURCE_CONFIG_ID}")
print(f"Resource Configuration ARN: {RESOURCE_CONFIG_ARN}")
print(f"Status:                     {rc_response['status']}")

In [ ]:
# Resource Configuration이 ACTIVE 상태가 될 때까지 대기
while True:
    rc = lattice_b.get_resource_configuration(resourceConfigurationIdentifier=RESOURCE_CONFIG_ID)
    status = rc["status"]
    print(f"Status: {status}")
    if status == "ACTIVE":
        print("\nResource Configuration is active!")
        break
    if status == "CREATE_FAILED":
        print(f"\nFailed: {rc}")
        break
    time.sleep(15)

## 4단계: AWS RAM을 통해 Resource Configuration 공유

계정 B는 AWS RAM을 사용하여 Resource Configuration을 계정 A와 공유합니다. 계정 A가 공유를 수락하면 계정 A에서 Resource Configuration ARN을 확인할 수 있습니다.

![AWS RAM 리소스 공유](./images/ram.png)

In [ ]:
# 계정 B: RAM 리소스 공유 생성
share_response = ram_b.create_resource_share(
    name="cross-account-rc-share",
    resourceArns=[RESOURCE_CONFIG_ARN],
    principals=[ACCOUNT_A_ID],
)

RAM_SHARE_ARN = share_response["resourceShare"]["resourceShareArn"]
print(f"RAM Share ARN: {RAM_SHARE_ARN}")
print(f"Status:        {share_response['resourceShare']['status']}")

In [ ]:
# 계정 A: RAM 공유 초대를 찾아 수락
time.sleep(5)  # 초대가 전파될 때까지 잠시 대기

invitations = ram_a.get_resource_share_invitations()["resourceShareInvitations"]

# 생성한 공유의 초대 찾기
invitation_arn = None
for inv in invitations:
    if inv["resourceShareArn"] == RAM_SHARE_ARN and inv["status"] == "PENDING":
        invitation_arn = inv["resourceShareInvitationArn"]
        break

if invitation_arn:
    accept = ram_a.accept_resource_share_invitation(resourceShareInvitationArn=invitation_arn)
    print("Accepted RAM share invitation")
    print(f"Status: {accept['resourceShareInvitation']['status']}")
else:
    print("No pending invitation found. It may have been auto-accepted (same org).")

print(f"\nResource Configuration ARN (shared): {RESOURCE_CONFIG_ARN}")
print("This ARN is now visible in Account A.")

## 5단계: 계정 A에 AgentCore Gateway 대상 생성

이제 계정 A에서 `selfManagedLatticeResource`를 사용하여 Gateway 대상을 생성합니다. 계정 B에서 **공유한 Resource Configuration ARN**을 전달합니다.

In [ ]:
# 계정 A에 API 키 자격 증명 공급자 생성
cred_response = agentcore.create_api_key_credential_provider(
    name="cross-account-apigw-api-key",
    apiKey=API_KEY_VALUE,
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

In [ ]:
# OpenAPI 스키마를 로드하고 서버 URL 설정
with open("02-self-managed-lattice/openapi-private-apigw.json") as f:
    openapi_schema = json.load(f)

TARGET_ENDPOINT = f"https://{API_VPCE_DNS}/prod"
openapi_schema["servers"] = [{"url": TARGET_ENDPOINT}]
OPENAPI_SCHEMA = json.dumps(openapi_schema)

print(f"Target endpoint:      {TARGET_ENDPOINT}")
print(f"Resource Config ARN:  {RESOURCE_CONFIG_ARN} (from Account B)")

response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="cross-account-apigw",
    description="Private API Gateway in Account B via self-managed VPC Lattice + RAM",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "selfManagedLatticeResource": {
            "resourceConfigurationIdentifier": RESOURCE_CONFIG_ARN,
        }
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Resource association: {target.get('privateEndpoint', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 6단계: AgentCore Gateway를 통해 API 호출

Cognito에서 액세스 토큰을 가져온 다음, 계정 A의 AgentCore Gateway를 통해 계정 B의 Private API Gateway를 호출합니다.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# 상태 확인 - 계정 B의 Private API Gateway에서 GET /health 호출
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "cross-account-apigw___healthCheck", "arguments": {}},
        "id": 1,
    },
)
print("Health check (Account B Private API Gateway via cross-account Lattice):")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 항목 목록 조회
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "cross-account-apigw___listItems", "arguments": {}},
        "id": 2,
    },
)
print("Items (from Account B):")
print(json.dumps(response.json(), indent=2))

## 정리

두 계정의 리소스를 역순으로 정리합니다.

1. **계정 A**: Gateway 대상 및 자격 증명 공급자 삭제
2. **계정 B**: RAM 공유 삭제
3. **계정 B**: Resource Configuration 및 Resource Gateway 삭제
4. **계정 B**: CDK 스택 제거 및 보존된 VPCE 보안 그룹 삭제

> **참고:** Gateway 대상을 삭제하면 AgentCore가 서비스 네트워크 리소스 연결을 비동기적으로 제거합니다. Resource Configuration 삭제가 "has existing association with service networks" 오류와 함께 실패하면 몇 분 기다린 후 다시 시도하세요.

In [ ]:
# # 1단계(계정 A): Gateway 대상 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # 자격 증명 공급자 삭제
# agentcore.delete_api_key_credential_provider(name="cross-account-apigw-api-key")
# print("Deleted credential provider")

In [ ]:
# # 2단계(계정 B): RAM 공유 삭제
# ram_b.delete_resource_share(resourceShareArn=RAM_SHARE_ARN)
# print(f"Deleted RAM share: {RAM_SHARE_ARN}")

In [ ]:
# # 3단계(계정 B): Resource Configuration 및 Resource Gateway 삭제
# # "existing association"으로 RC 삭제가 실패하면 몇 분 기다린 후 이 셀을 다시 실행합니다.
# try:
#     lattice_b.delete_resource_configuration(resourceConfigurationIdentifier=RESOURCE_CONFIG_ID)
#     print(f"Deleted Resource Configuration: {RESOURCE_CONFIG_ID}")

#     lattice_b.delete_resource_gateway(resourceG`atewayIdentifier=RESOURCE_GATEWAY_ID)
#     print(f"Deleted Resource Gateway: {RESOURCE_GATEWAY_ID}")`
# except lattice_b.exceptions.ClientError as e:
#     if "existing association with service networks" in str(e):
#         print(f"RC still associated with service network. Wait a few minutes and re-run this cell.")
#     else:
#         raise

In [ ]:
# # 4단계(계정 B): CDK 스택 제거 및 보존된 VPCE 보안 그룹 삭제
# !ACCOUNT_B_ID={ACCOUNT_B_ID} cdk destroy CrossAccountApigw-AccountB --profile {ACCOUNT_B_PROFILE} --force

# # 보존된 VPCE 보안 그룹 삭제
# ec2_b = session_b.client("ec2")
# try:
#     ec2_b.delete_security_group(GroupId=CROSSACCT_VPCE_SG_ID)
#     print(f"Deleted VPCE security group: {CROSSACCT_VPCE_SG_ID}")
# except ec2_b.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(f"SG {CROSSACCT_VPCE_SG_ID} still has dependencies. Wait a few minutes and retry.")
#     else:
#         raise